# 9 · Validation (Mass, Inertia, Simulation)

**Author:** Héctor Fernández Pinacho  
**Lab:** IDEAL Lab · ETH Zürich

The full validation of the pipeline against physical measurements, in three stages. **Stage 1 (mass):** STL-predicted mass versus the precision-scale measurement, for every measured propeller. **Stage 2 (inertia):** STL-predicted spin-axis inertia versus the trifilar-pendulum measurement, raw and regression-corrected. **Stage 3 (flight):** the simulated peak flight height — both the aero-only NB6 prediction and the release-corrected NB6b prediction — versus the free-flight launcher, over all 30 flight-tested propellers (the 20 tested representative props plus the 10 recovered validation props whose `val_` outputs come from NB3–NB6b). Only PASS-flagged, spike-filtered launcher runs are used; cells that reached the string ceiling are right-censored and excluded from the quantitative metrics but scored separately for qualitative agreement. The notebook closes by assembling the **validated dataset** (`csv/dataset_validated.csv`, the measured counterpart of the full simulation dataset from NB6b). The high-resolution results figures are generated centrally by NB8.

**Physics inputs:** `csv/00_measured_mass_inertia.csv` and `csv/00_validation_geometry.csv` (via `utils/measurements.py`), `stl/` meshes, the cleaned launcher runs in `results/*_cleaned/`, `csv/07_selected.csv`, the per-RPM flight predictions `csv/06_flight_dynamics_*rpm.csv`, `csv/06b_flight_dynamics_release_*rpm.csv` and their `val_` equivalents, `csv/06b_release_retention_curve.csv`, `csv/05_qprop_sweep.csv.gz`

**Physics outputs:** the validation tables `csv/validation_mass_inertia.csv`, `csv/validation_sim_matched.csv`, `csv/validation_sim_ranking_per_config.csv`, `csv/validation_master_summary.csv`, `csv/validation_per_rpm_summary.csv`, `csv/validation_secondary_summary.csv`, the validated dataset `csv/dataset_validated.csv`, and the diagnostic figures in `plots/nb9/`

**Structure:**

1. Imports
2. Configuration — all tunable constants, paths and settings
3. Function definitions — error reporting, trifilar conversion, STL shape extraction, launcher-run cleaning, aggregation, simulation loading and the analysis blocks
4. Main code — density fit, the three validation stages, summaries, plots and the validated dataset



## 1. Imports

In [ ]:
import os
import sys

os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
sys.dont_write_bytecode = True

import glob
import math
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from scipy.interpolate import PchipInterpolator

try:
    import trimesh
except ImportError as import_error:
    raise ImportError('This notebook requires trimesh (pip install trimesh).') from import_error

BASE_DIR = Path(globals().get('__vsc_ipynb_file__', Path.cwd())).parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
import pipeline_config as cfg
from utils import measurements

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

## 2. Configuration

In [ ]:
CSV_DIR = BASE_DIR / 'csv'
STL_DIR = BASE_DIR / 'stl'
RESULTS_DIR = BASE_DIR / 'results'
PLOTS_DIR = BASE_DIR / 'plots' / 'nb9'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

MEASURED_MASS_INERTIA_CSV = measurements.measured_mass_inertia_path(BASE_DIR)

GRAVITY_M_S2 = 9.80665
PENDULUM_RADIUS_M = 0.090
PENDULUM_STRING_M = 0.700
EMPTY_PLATE_MASS_KG = 11.7 / 1000
EMPTY_PLATE_TIME_10OSC_S = 9.2
OSCILLATION_COUNT = 10

STL_FILENAME_PATTERNS = ['prop_{config_id}.stl', 'config_{config_id}.stl', '{config_id}.stl']

SELECTED_CSV = CSV_DIR / cfg.CSV_NAMES['selected']
GEOMETRY_CSV = CSV_DIR / cfg.CSV_NAMES['geometry']

VALIDATE_SIM = 'release'
SIM_AERO_PATTERNS = ['06_flight_dynamics_*rpm.csv', 'val_06_flight_dynamics_*rpm.csv']
SIM_RELEASE_PATTERNS = ['06b_flight_dynamics_release_*rpm.csv', 'val_06b_flight_dynamics_release_*rpm.csv']

INCLUDE_VALIDATION_PROPS = measurements.validation_geometry_path(BASE_DIR).exists() and any(CSV_DIR.glob('val_06b_flight_dynamics_release_*rpm.csv'))
if INCLUDE_VALIDATION_PROPS:
    print('  Validation-subset props    : INCLUDED (30 props)')
else:
    print('  Validation-subset props    : not present (20 props)')

USE_REGRESSED_TARGET = True
if USE_REGRESSED_TARGET:
    print('  Measured target            : isotonic-regressed PASS-mean')
else:
    print('  Measured target            : raw per-RPM max/median')

USE_RPM_CONFIRMED_PEAKS = True
RPM_FREEZE_WINDOW = 15
RPM_FREEZE_TOL_RPM = 1.0
MAX_CLIMB_SPEED_MS = 8.0
PEAK_DWELL_S = 0.05
PEAK_NEAR_FRACTION = 0.70
if USE_RPM_CONFIRMED_PEAKS:
    print('  RPM-confirmed peaks        : ON (frozen-RPM + kinematic glitch rejection)')
else:
    print('  RPM-confirmed peaks        : OFF')

TEST_REPORTS = [
    RESULTS_DIR / '01_PropellerTesting_FirstBatch_cleaned' / 'cleaned_validation_report.csv',
    RESULTS_DIR / '02_PropellerTesting_SecondBatch_cleaned' / 'cleaned_validation_report.csv',
]
TEST_CLEANED_DIRS = []
for report_path in TEST_REPORTS:
    TEST_CLEANED_DIRS.append(report_path.parent)

LIFTOFF_HEIGHT_M = 0.05
HEIGHT_CEILING_M = 2.60
CENSOR_AT_M = 2.40
V_ENVELOPE = 5.0
KICK_VELOCITY_MS = 1.9
H_KICK_FLOOR = KICK_VELOCITY_MS ** 2 / (2 * 9.81)

print('Validation notebook configuration')
print(f'  Measured mass/inertia CSV : {MEASURED_MASS_INERTIA_CSV}  (exists={MEASURED_MASS_INERTIA_CSV.exists()})')
report_flags = []
for report_path in TEST_REPORTS:
    report_flags.append(report_path.exists())
print(f'  Validating simulation      : {VALIDATE_SIM}')
print(f'  Test reports present       : {report_flags}')

## 3. Function Definitions

**Reporting and physics helpers**

- **error_metrics_report(config_ids, measured, predicted, label, unit, percent)** — prints the standard error metrics (bias, mean absolute error, RMSE) for measured-vs-predicted values plus the single best and worst prop, and returns them as a dict.
- **trifilar_inertia_kg_m2(time_for_oscillations_s, oscillating_mass_kg)** — converts a measured trifilar-pendulum oscillation time into a spin-axis moment of inertia using the rig constants.
- **find_stl_path(config_id)** — locates a config's STL inside `stl/`.
- **stl_shape_properties(config_id)** — reads one STL and returns its volume and density-independent spin-axis inertia coefficient (later scaled by the fitted print density).
- **percent_error_report(config_ids, percent_errors, label)** — the percent-error twin of the metrics report.
- **ranking_metrics(predicted, measured)** — Spearman ρ, Kendall τ and Pearson r with NaN-safe guards.
- **smooth_monotone_curve(rpm, height, n_points)** — PCHIP (monotone cubic Hermite) curve through an isotonic height-vs-RPM series, for plotting the regressed target without staircase corners.
- **signed_fmt(value) / plain_fmt(value)** — table formatters.

**Launcher-run cleaning and simulation loading**

- **parse_run_filename(filename)** — splits `PROPID_BLADENR_TARGETRPM_RUNNR(_cleaned).csv` into its four fields.
- **read_run_trace(cleaned_dir, config_id, blades, rpm_launch, run)** — loads one cleaned run's (time, RPM, height) trace.
- **kinematic_deglitch(time_s, height_m, max_v)** — drops samples that rose faster than the kinematic bound and do not stay elevated afterwards (transient sensor jumps), returning a keep-mask.
- **rpm_confirmed_peak(trace, window, tol)** — the highest peak that survives all physical checks: kinematic de-glitch, apex dwell, and the frozen-RPM dual-sensor-latch rejection; falls back to the next lower peak.
- **load_pass_runs(report_paths)** — reads the cleaned validation reports, keeps only PASS runs, and re-reads each run's trace to apply the RPM-confirmed peak cleaning.
- **isotonic_increasing(rpm, height, weight)** — monotone non-decreasing least-squares fit via the pool-adjacent-violators algorithm (pure NumPy).
- **aggregate_pass_cells(pass_runs)** — one row per (config, RPM) with max/median/mean/min height, run count, censoring and launch flags, plus the isotonic-regressed target.
- **load_sim_long(csv_dir, patterns, height_label)** — stacks the per-RPM simulated flight files (main and `val_` patterns) into one long table, preferring the validation rows for the 10 recovered ids.
- **index_run_trajectories(cleaned_dirs)** — indexes the cleaned run files by (config, RPM) for the trajectory grids.

**Analysis blocks** (each takes the cell table it analyses as its first argument)

- **fit_release_AB(d)** — OLS of measured median height on the aero height; returns (A, B).
- **global_row(frame, sim_col, sim_name, target_col) / per_rpm_table(frame, sim_col, target_col) / summary_block(frame, sim_col, target_col, only_flying)** — pooled and per-RPM ranking + error blocks for any sim/target combination.
- **scatter_panel(ax, frame, all_cells, sim_col, target_col, title, only_flying)** — one sim-vs-measured scatter with the 1:1 line, RPM colouring and censored markers (censored cells drawn from `all_cells`).
- **per_rpm_rank(frame, sim_col, target_col) / per_rpm_err(frame, sim_col, target_col)** — ranking and error curves versus RPM for the overview figure.
- **make_grid(df, value_col)** — pivots sweep rows into a plot-ready surface grid.
- **cell_is_clean(sweep_all, config_id, launch_rpm) / void_block(frame, sim_col, target_col)** — the void-free-surface subset check and its metric block.


In [ ]:
def error_metrics_report(config_ids, measured, predicted, label, unit, percent=False):

    config_ids = np.asarray(config_ids)
    measured = np.asarray(measured, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    keep = np.isfinite(measured) & np.isfinite(predicted)
    config_ids = config_ids[keep]
    measured = measured[keep]
    predicted = predicted[keep]

    error = predicted - measured
    abs_error = np.abs(error)
    mean_error = float(np.mean(error))
    mean_abs_err = float(np.mean(abs_error))
    rmse = float(np.sqrt(np.mean(error ** 2)))

    best_index = int(np.argmin(abs_error))
    worst_index = int(np.argmax(abs_error))

    if len(measured):
        scale = np.nanmax(np.abs(measured))
    else:
        scale = 1.0
    if scale < 0.01:
        fmt = '{:+.4g}'
        fmtp = '{:.4g}'
    else:
        fmt = '{:+.4f}'
        fmtp = '{:.4f}'

    print(f'--- {label} ({len(measured)} props) ---')
    print(f'  mean error (bias)      : {fmt.format(mean_error)} {unit}')
    print(f'  mean absolute error    : {fmtp.format(mean_abs_err)} {unit}')
    print(f'  RMSE                   : {fmtp.format(rmse)} {unit}')
    print(f'  BEST  (smallest error) : config {int(config_ids[best_index])}  measured={fmtp.format(measured[best_index])}  predicted={fmtp.format(predicted[best_index])}  error={fmt.format(error[best_index])} {unit}')
    print(f'  WORST (largest error)  : config {int(config_ids[worst_index])}  measured={fmtp.format(measured[worst_index])}  predicted={fmtp.format(predicted[worst_index])}  error={fmt.format(error[worst_index])} {unit}')

    return {
        'label': label, 'n': int(len(measured)),
        'mean_error': mean_error, 'mean_abs_error': mean_abs_err, 'rmse': rmse,
        'best_config': int(config_ids[best_index]), 'best_abs_error': float(abs_error[best_index]),
        'worst_config': int(config_ids[worst_index]), 'worst_abs_error': float(abs_error[worst_index]),
    }


def trifilar_inertia_kg_m2(time_for_oscillations_s, oscillating_mass_kg):

    period_s = time_for_oscillations_s / OSCILLATION_COUNT

    return (period_s ** 2) * oscillating_mass_kg * GRAVITY_M_S2 * (PENDULUM_RADIUS_M ** 2) / (4 * math.pi ** 2 * PENDULUM_STRING_M)


def find_stl_path(config_id):

    config_id = int(config_id)
    for pattern in STL_FILENAME_PATTERNS:
        candidate = STL_DIR / pattern.format(config_id=config_id)
        if candidate.exists():

            return candidate
    matches = sorted(STL_DIR.glob(f'*{config_id}*.stl'))
    if matches:

        return matches[0]

    return None


def stl_shape_properties(config_id):

    path = find_stl_path(config_id)
    if path is None:

        return {'config_id': int(config_id), 'stl_found': False}

    try:
        mesh = trimesh.load_mesh(path, force='mesh')
        if isinstance(mesh, trimesh.Scene):
            mesh = trimesh.util.concatenate(tuple(mesh.dump()))
        volume_mm3 = abs(float(mesh.volume))
        center_of_mass = np.asarray(mesh.center_mass, dtype=float)

        unit_density_mesh = mesh.copy()
        unit_density_mesh.density = 1.0
        unit_mass_kg = abs(float(unit_density_mesh.mass))
        inertia_about_com = float(np.asarray(unit_density_mesh.moment_inertia)[2, 2])
        inertia_about_axis = inertia_about_com + unit_mass_kg * (center_of_mass[0] ** 2 + center_of_mass[1] ** 2)

        return {
            'config_id': int(config_id), 'stl_found': True,
            'volume_mm3': volume_mm3, 'volume_cm3': volume_mm3 / 1000.0,
            'izz_unit_kg_mm2': inertia_about_axis,
        }
    except Exception as load_error:

        return {'config_id': int(config_id), 'stl_found': True, 'error': repr(load_error)}


def percent_error_report(config_ids, percent_errors, label):

    config_ids = np.asarray(config_ids)
    percent_errors = np.asarray(percent_errors, dtype=float)
    keep = np.isfinite(percent_errors)
    config_ids = config_ids[keep]
    percent_errors = percent_errors[keep]
    abs_pct = np.abs(percent_errors)
    best_index = int(np.argmin(abs_pct))
    worst_index = int(np.argmax(abs_pct))

    print(f'--- {label} ({len(percent_errors)} props) ---')
    print(f'  mean percent error (bias) : {float(np.mean(percent_errors)):+.2f} %')
    print(f'  mean ABS percent error    : {float(np.mean(abs_pct)):.2f} %')
    print(f'  median ABS percent error  : {float(np.median(abs_pct)):.2f} %')
    print(f'  BEST  (smallest |error|)  : config {int(config_ids[best_index])}  ({percent_errors[best_index]:+.2f} %)')
    print(f'  WORST (largest |error|)   : config {int(config_ids[worst_index])}  ({percent_errors[worst_index]:+.2f} %)')

    return {'label': label, 'n': int(len(percent_errors)),
            'mean_pct': float(np.mean(percent_errors)), 'mean_abs_pct': float(np.mean(abs_pct)),
            'best_config': int(config_ids[best_index]), 'worst_config': int(config_ids[worst_index])}


def ranking_metrics(predicted, measured):

    predicted = np.asarray(predicted, float)
    measured = np.asarray(measured, float)
    keep = np.isfinite(predicted) & np.isfinite(measured)
    predicted = predicted[keep]
    measured = measured[keep]
    n = len(predicted)
    if n < 3 or np.std(predicted) == 0 or np.std(measured) == 0:

        return dict(n=n, spearman=np.nan, pearson=np.nan, kendall=np.nan)

    with warnings.catch_warnings():
        warnings.simplefilter('ignore')

        return dict(
            n=n,
            spearman=float(stats.spearmanr(predicted, measured).statistic),
            pearson=float(stats.pearsonr(predicted, measured)[0]),
            kendall=float(stats.kendalltau(predicted, measured).statistic),
        )


def smooth_monotone_curve(rpm, height, n_points=200):

    rpm = np.asarray(rpm, float)
    height = np.asarray(height, float)
    order = np.argsort(rpm)
    rpm = rpm[order]
    height = height[order]
    keep = np.concatenate([[True], np.diff(rpm) > 0])
    rpm = rpm[keep]
    height = height[keep]
    if len(rpm) < 2:

        return rpm, height

    rpm_dense = np.linspace(rpm.min(), rpm.max(), n_points)
    height_dense = PchipInterpolator(rpm, height)(rpm_dense)

    return rpm_dense, height_dense


def signed_fmt(value):

    if pd.notna(value):
        text = f'{value:+.3f}'
    else:
        text = '—'

    return text


def plain_fmt(value):

    if pd.notna(value):
        text = f'{value:.3f}'
    else:
        text = '—'

    return text

In [ ]:
def parse_run_filename(filename):

    match = re.match(r'(\d+)_(\d+)_(\d+)_(\d+)', Path(filename).name)
    if not match:

        return (None, None, None, None)

    return int(match.group(1)), int(match.group(2)), int(match.group(3)), int(match.group(4))


def read_run_trace(cleaned_dir, config_id, blades, rpm_launch, run):

    name = f'{config_id}_{blades}_{rpm_launch}_{run}_cleaned.csv'
    path = Path(cleaned_dir) / name
    if not path.exists():

        return None

    times = []
    rpms = []
    heights = []
    with open(path) as run_file:
        for line in run_file:
            if line.startswith('#') or line.startswith('time_s'):
                continue
            parts = line.strip().split(',')
            if len(parts) < 3:
                continue
            try:
                time_value = float(parts[0])
                if parts[1]:
                    rpm_value = float(parts[1])
                else:
                    rpm_value = np.nan
                height_value = float(parts[2])
            except ValueError:
                continue
            times.append(time_value)
            rpms.append(rpm_value)
            heights.append(height_value)
    if not times:

        return None

    return pd.DataFrame({'t': times, 'rpm': rpms, 'h': heights})


def kinematic_deglitch(time_s, height_m, max_v=None):

    if max_v is None:
        max_v = MAX_CLIMB_SPEED_MS
    keep = np.ones(len(height_m), dtype=bool)
    last = 0
    for i in range(1, len(height_m)):
        dt = time_s[i] - time_s[last]
        if dt <= 0:
            keep[i] = False
            continue
        rise = height_m[i] - height_m[last]
        if rise > max_v * dt + 0.05:
            ahead = (time_s > time_s[i]) & (time_s <= time_s[i] + 0.10)
            if ahead.sum() >= 2 and np.nanmedian(height_m[ahead]) >= height_m[last] + 0.5 * rise:
                last = i
            else:
                keep[i] = False
        else:
            last = i

    return keep


def rpm_confirmed_peak(trace, window=None, tol=None):

    if window is None:
        window = RPM_FREEZE_WINDOW
    if tol is None:
        tol = RPM_FREEZE_TOL_RPM
    if trace is None or trace['h'].isna().all():

        return np.nan

    t = trace['t'].values
    h = trace['h'].values
    rpm = trace['rpm'].values
    keep = kinematic_deglitch(t, h)
    tk = t[keep]
    hk = h[keep]
    rk = rpm[keep]
    if len(hk) == 0:

        return 0.0

    for ipk in np.argsort(hk)[::-1]:
        if hk[ipk] <= LIFTOFF_HEIGHT_M:

            return max(float(hk[ipk]), 0.0)

        near = hk >= PEAK_NEAR_FRACTION * hk[ipk]
        lo = ipk
        hi = ipk
        while lo > 0 and near[lo - 1]:
            lo -= 1
        while hi < len(hk) - 1 and near[hi + 1]:
            hi += 1
        if (tk[hi] - tk[lo]) < PEAK_DWELL_S:
            continue
        before = pd.Series(rk[max(0, ipk - window):ipk]).dropna()
        after = pd.Series(rk[ipk + 1:ipk + 1 + window]).dropna()
        before_frozen = len(before) >= 4 and before.std() < tol
        after_frozen = len(after) >= 4 and after.std() < tol
        if not (before_frozen and after_frozen):

            return float(hk[ipk])

    return 0.0


def load_pass_runs(report_paths):

    frames = []
    for report_path in report_paths:
        if not Path(report_path).exists():
            print(f'  [WARN] missing report: {report_path}')
            continue
        frames.append(pd.read_csv(report_path))
    if not frames:
        raise FileNotFoundError('No cleaned validation reports found under results/.')

    runs = pd.concat(frames, ignore_index=True)
    parsed_rows = []
    for name in runs['file']:
        parsed_rows.append(parse_run_filename(name))
    parsed = pd.DataFrame(parsed_rows, columns=['config_id', 'blades', 'rpm_launch', 'run'], index=runs.index)
    runs = pd.concat([runs, parsed], axis=1).dropna(subset=['config_id', 'rpm_launch'])
    runs[['config_id', 'blades', 'rpm_launch', 'run']] = runs[['config_id', 'blades', 'rpm_launch', 'run']].astype(int)
    runs['max_height_m'] = pd.to_numeric(runs.get('max_height_m'), errors='coerce')
    runs['status'] = runs['status'].astype(str).str.upper().str.strip()

    n_total = len(runs)
    pass_runs = runs[(runs['status'] == 'PASS') & runs['max_height_m'].notna()].copy()
    print(f'  Runs in reports : {n_total}')
    print(f"  Status counts   : {runs['status'].value_counts().to_dict()}")
    print(f'  Kept (PASS)     : {len(pass_runs)}')

    if USE_RPM_CONFIRMED_PEAKS:
        pass_runs['naive_height_m'] = pass_runs['max_height_m']
        confirmed = []
        for row in pass_runs.itertuples(index=False):
            trace = None
            for cleaned_dir in TEST_CLEANED_DIRS:
                trace = read_run_trace(cleaned_dir, row.config_id, row.blades, row.rpm_launch, row.run)
                if trace is not None:
                    break
            if trace is not None:
                confirmed.append(rpm_confirmed_peak(trace))
            else:
                confirmed.append(row.max_height_m)
        pass_runs['max_height_m'] = confirmed
        n_changed = int((np.abs(pass_runs['naive_height_m'] - pass_runs['max_height_m']) > 0.1).sum())
        n_spike = int(((pass_runs['naive_height_m'] > 1.0) & (pass_runs['max_height_m'] < 0.1)).sum())
        print(f'  RPM-confirmed     : corrected {n_changed} runs ({n_spike} high spikes dropped to ~0); peaks with frozen RPM both sides rejected')

    return pass_runs


def isotonic_increasing(rpm, height, weight=None):

    rpm = np.asarray(rpm, float)
    height = np.asarray(height, float)
    if weight is None:
        weight = np.ones_like(height)
    else:
        weight = np.asarray(weight, float)
    order = np.argsort(rpm)
    y = height[order]
    w = weight[order]
    blocks = []
    for value_i, weight_i in zip(y, w):
        blocks.append([value_i, weight_i, 1])
        while len(blocks) >= 2 and blocks[-2][0] > blocks[-1][0]:
            v2, w2, c2 = blocks.pop()
            v1, w1, c1 = blocks.pop()
            merged_w = w1 + w2
            blocks.append([(v1 * w1 + v2 * w2) / merged_w, merged_w, c1 + c2])
    fitted_pieces = []
    for value, weight_unused, count in blocks:
        fitted_pieces.append(np.full(count, value))
    fitted_sorted = np.concatenate(fitted_pieces)
    fitted = np.empty_like(fitted_sorted)
    fitted[order] = fitted_sorted

    return fitted


def aggregate_pass_cells(pass_runs):

    agg = pass_runs.groupby(['config_id', 'rpm_launch']).agg(
        meas_h_max=('max_height_m', 'max'),
        meas_h_median=('max_height_m', 'median'),
        meas_h_mean=('max_height_m', 'mean'),
        meas_h_min=('max_height_m', 'min'),
        n_pass_runs=('max_height_m', 'size'),
    ).reset_index()
    agg['meas_censored'] = agg['meas_h_max'] >= CENSOR_AT_M
    agg['meas_launch'] = agg['meas_h_max'] >= LIFTOFF_HEIGHT_M

    regressed = []
    for cid, group in agg.groupby('config_id'):
        group = group.sort_values('rpm_launch').copy()
        if len(group) >= 3:
            group['meas_h_reg'] = isotonic_increasing(group['rpm_launch'], group['meas_h_mean'], group['n_pass_runs'])
        else:
            group['meas_h_reg'] = group['meas_h_mean'].values
        regressed.append(group)
    agg = pd.concat(regressed, ignore_index=True)

    return agg


def load_sim_long(csv_dir, patterns, height_label):

    frames = []
    for pattern in patterns:
        for file_path in sorted(Path(csv_dir).glob(pattern)):
            rpm_match = re.search(r'_(\d+)rpm', file_path.name)
            if not rpm_match:
                continue
            frame = pd.read_csv(file_path)
            frame['rpm_launch'] = int(rpm_match.group(1))
            keep = []
            for column in ['config_id', 'rpm_launch', 'h_max_m', 'h_max_aero_m', 'can_liftoff', 'flight_ok']:
                if column in frame.columns:
                    keep.append(column)
            frames.append(frame[keep])
    sim = pd.concat(frames, ignore_index=True).dropna(subset=['config_id'])
    sim['config_id'] = sim['config_id'].astype(int)
    sim = sim.drop_duplicates(subset=['config_id', 'rpm_launch'], keep='last')

    return sim.rename(columns={'h_max_m': height_label})


def index_run_trajectories(cleaned_dirs):

    idx = {}
    for cleaned_dir in cleaned_dirs:
        for run_path in glob.glob(str(Path(cleaned_dir) / '*_cleaned.csv')):
            cid, blades_unused, rpm, run = parse_run_filename(run_path)
            if cid is None:
                continue
            idx.setdefault((cid, rpm), []).append(run_path)

    return idx

In [ ]:
def fit_release_AB(d):

    B_slope, A_int = np.polyfit(d['h_aero_m'].to_numpy(float), d['meas_h_median'].to_numpy(float), 1)

    return float(A_int), float(B_slope)


def global_row(frame, sim_col, sim_name, target_col):

    cells = frame.dropna(subset=[sim_col, target_col])
    if sim_col == 'h_aero_m':
        cells = cells[cells[sim_col] > 0]
    rank = ranking_metrics(cells[sim_col], cells[target_col])
    err = cells[sim_col] - cells[target_col]

    return {'sim': sim_name, 'target': target_col.replace('meas_h_', ''),
            'n': len(cells), 'spearman': rank['spearman'], 'kendall': rank['kendall'],
            'pearson': rank['pearson'], 'MAE_m': float(np.mean(np.abs(err))),
            'RMSE_m': float(np.sqrt(np.mean(err ** 2))), 'bias_m': float(np.mean(err))}


def per_rpm_table(frame, sim_col, target_col):

    rows = []
    for rpm, group in frame.groupby('rpm_launch'):
        cells = group.dropna(subset=[sim_col, target_col])
        if sim_col == 'h_aero_m':
            cells = cells[cells[sim_col] > 0]
        if cells['config_id'].nunique() < 3:
            continue
        rank = ranking_metrics(cells[sim_col], cells[target_col])
        err = cells[sim_col] - cells[target_col]
        rows.append({'rpm_launch': int(rpm), 'n_props': rank['n'],
                     'spearman': rank['spearman'], 'kendall': rank['kendall'],
                     'MAE_m': float(np.mean(np.abs(err))), 'bias_m': float(np.mean(err))})

    return pd.DataFrame(rows)


def summary_block(frame, sim_col, target_col, only_flying=False):

    d = frame.dropna(subset=[sim_col, target_col])
    if only_flying:
        d = d[d[sim_col] > 0]
    if len(d) < 3:

        return dict(n=len(d), spearman=np.nan, kendall=np.nan, pearson=np.nan, MAE_m=np.nan, RMSE_m=np.nan, bias_m=np.nan)

    rk = ranking_metrics(d[sim_col], d[target_col])
    err = d[sim_col] - d[target_col]

    return dict(n=len(d), spearman=rk['spearman'], kendall=rk['kendall'], pearson=rk['pearson'],
                MAE_m=float(np.mean(np.abs(err))), RMSE_m=float(np.sqrt(np.mean(err ** 2))), bias_m=float(np.mean(err)))


def scatter_panel(ax, frame, all_cells, sim_col, target_col, title, only_flying=False):

    data = frame.dropna(subset=[sim_col, target_col]).copy()
    if only_flying:
        data = data[data[sim_col] > 0]
    if len(data) < 3:
        ax.set_title(title + '\n(insufficient data)')
        ax.axis('off')

        return None

    rho = stats.spearmanr(data[sim_col], data[target_col]).statistic
    sc = ax.scatter(data[target_col], data[sim_col], c=data['rpm_launch'], cmap='viridis', s=42, edgecolor='k', linewidth=0.3, zorder=3)

    censored = all_cells[all_cells['meas_censored'] & all_cells['meas_launch']].dropna(subset=[sim_col]).copy()
    if only_flying:
        censored = censored[censored[sim_col] > 0]
    if len(censored):
        ax.scatter(np.full(len(censored), HEIGHT_CEILING_M), censored[sim_col], facecolors='none', edgecolors='red', marker='>', s=55, linewidth=1.2, zorder=4, label=f'censored (≥ ceiling, n={len(censored)})')
        ax.axvline(HEIGHT_CEILING_M, color='red', ls=':', lw=0.9, alpha=0.6)

    lim = float(np.nanmax([data[target_col].max(), data[sim_col].max(), HEIGHT_CEILING_M, 0.5])) * 1.05
    ax.plot([0, lim], [0, lim], 'k--', lw=1, label='1:1')
    ax.set_xlim(0, lim)
    ax.set_ylim(0, lim)
    ax.set_xlabel(f"measured {target_col.replace('meas_h_', '')} height [m]")
    ax.set_ylabel('simulated height [m]')
    ax.set_title(f'{title}\nSpearman ρ = {rho:+.2f}  |  n = {len(data)}', fontsize=10)
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

    return sc


def per_rpm_rank(frame, sim_col, target_col):

    rows = []
    for rpm, g in frame.groupby('rpm_launch'):
        d = g.dropna(subset=[sim_col, target_col])
        if sim_col == 'h_aero_m':
            d = d[d[sim_col] > 0]
        if d['config_id'].nunique() >= 4 and d[sim_col].nunique() > 2 and d[target_col].nunique() > 1:
            rows.append({'rpm': int(rpm),
                         'spearman': stats.spearmanr(d[sim_col], d[target_col]).statistic,
                         'kendall': stats.kendalltau(d[sim_col], d[target_col]).statistic})

    return pd.DataFrame(rows)


def per_rpm_err(frame, sim_col, target_col):

    rows = []
    for rpm, g in frame.groupby('rpm_launch'):
        d = g.dropna(subset=[sim_col, target_col])
        if sim_col == 'h_aero_m':
            d = d[d[sim_col] > 0]
        if len(d):
            err = d[sim_col] - d[target_col]
            rows.append({'rpm': int(rpm), 'mae': np.mean(np.abs(err)), 'bias': np.mean(err)})

    return pd.DataFrame(rows)


def make_grid(df, value_col):

    pivot = df.pivot_table(index='V', columns='rpm', values=value_col, aggfunc='mean')
    rpm_mesh, V_mesh = np.meshgrid(pivot.columns.values, pivot.index.values)

    return rpm_mesh, V_mesh, pivot.values


def cell_is_clean(sweep_all, config_id, launch_rpm):

    g = sweep_all[sweep_all['config_id'] == config_id]
    if g.empty:

        return False

    region = g[(g['V'] <= V_ENVELOPE) & (g['rpm'] <= launch_rpm)]
    V_axis = np.sort(region['V'].unique())
    rpm_axis = np.sort(region['rpm'].unique())
    if len(V_axis) == 0 or len(rpm_axis) == 0:

        return False

    present = region.dropna(subset=['T', 'Q']).drop_duplicates(['V', 'rpm']).shape[0]

    return present == len(V_axis) * len(rpm_axis)


def void_block(frame, sim_col, target_col):

    d = frame.dropna(subset=[sim_col, target_col])
    if sim_col == 'h_aero_m':
        d = d[d[sim_col] > 0]
    if len(d) < 3:

        return None

    rk = ranking_metrics(d[sim_col], d[target_col])
    err = d[sim_col] - d[target_col]

    return dict(n=len(d), spearman=rk['spearman'], MAE_m=float(np.mean(np.abs(err))), bias_m=float(np.mean(err)))

## 4. Main Code

The main code fits the print density on the measured props, runs the mass and inertia validations, builds the matched simulation-vs-measurement table over all 30 tested props, computes the global, per-RPM and per-config readings with the leave-one-prop-out cross-validation of the release correction, checks the censored cells, the void-free subset and the liftoff classification, saves the validation tables, generates the diagnostic figures, and assembles the validated dataset.


### 4.1 Load Measurements and Fit the Print Density

Loads the measured mass and trifilar period for every bench-tested prop (through `utils/measurements.py`), converts the period into a measured inertia, reads each prop's STL shape, fits the single effective SLS print density that maps STL volume to mass, and refits the linear inertia correction on this notebook's own validation props so the corrected curve is self-consistent.


In [ ]:
measured = measurements.load_measured_mass_inertia(BASE_DIR)

empty_plate_inertia = trifilar_inertia_kg_m2(EMPTY_PLATE_TIME_10OSC_S, EMPTY_PLATE_MASS_KG)
measured['mass_kg_meas'] = measured['mass_g'] / 1000.0
izz_values = []
for row in measured.itertuples(index=False):
    izz_values.append(trifilar_inertia_kg_m2(row.T_meas, EMPTY_PLATE_MASS_KG + row.mass_kg_meas) - empty_plate_inertia)
measured['izz_meas_kg_m2'] = izz_values

measured_by_prop = measured.groupby('config_id', as_index=False).agg(mass_g_meas=('mass_g', 'mean'), izz_meas_kg_m2=('izz_meas_kg_m2', 'mean'), n_measurements=('mass_g', 'size'))

shape_rows = []
for cid in measured_by_prop['config_id']:
    shape_rows.append(stl_shape_properties(cid))
stl_shapes = pd.DataFrame(shape_rows)
stl_shapes = stl_shapes[stl_shapes['stl_found'].fillna(False)].dropna(subset=['volume_cm3'])

validation = measured_by_prop.merge(stl_shapes, on='config_id', how='inner')
if validation.empty:
    raise RuntimeError('No measured props have a matching STL in ./stl.')

effective_density_g_cm3 = validation['mass_g_meas'].sum() / validation['volume_cm3'].sum()
effective_density_kg_mm3 = effective_density_g_cm3 * 1e-6

validation['mass_g_pred'] = validation['volume_mm3'] * effective_density_kg_mm3 * 1000.0
validation['izz_pred_kg_m2'] = validation['izz_unit_kg_mm2'] * effective_density_kg_mm3 * 1e-6

izz_fit_mask = validation['izz_meas_kg_m2'].notna() & validation['izz_pred_kg_m2'].notna()
izz_slope, izz_intercept = np.polyfit(validation.loc[izz_fit_mask, 'izz_pred_kg_m2'], validation.loc[izz_fit_mask, 'izz_meas_kg_m2'], 1)
validation['izz_pred_regressed_kg_m2'] = izz_slope * validation['izz_pred_kg_m2'] + izz_intercept
print(f'Inertia correction (fit here): izz_corrected = {izz_slope:.5f} * izz_STL + {izz_intercept:+.3e}')

validation['mass_error_pct'] = (validation['mass_g_meas'] - validation['mass_g_pred']) / validation['mass_g_pred'] * 100.0
validation['izz_error_pct'] = (validation['izz_meas_kg_m2'] - validation['izz_pred_kg_m2']) / validation['izz_pred_kg_m2'] * 100.0
validation['izz_error_pct_regressed'] = (validation['izz_meas_kg_m2'] - validation['izz_pred_regressed_kg_m2']) / validation['izz_pred_regressed_kg_m2'] * 100.0

print(f'Measured props loaded        : {len(measured_by_prop)}')
print(f'Matched to an STL            : {len(validation)}')
print(f'Fitted effective density     : {effective_density_g_cm3:.4f} g/cm^3')
validation[['config_id', 'mass_g_meas', 'mass_g_pred', 'izz_meas_kg_m2', 'izz_pred_kg_m2']].head()

### 4.2 Stage 1 — Mass Validation

Predicted mass = STL volume × fitted print density, compared to the precision-scale measurement for every measured propeller, followed by the predicted-vs-measured, residual and per-prop percent-error figures.


In [ ]:
print('=' * 64)
print('STAGE 1 - MASS VALIDATION')
print('=' * 64)

mass_abs = error_metrics_report(validation['config_id'], validation['mass_g_meas'], validation['mass_g_pred'], label='Mass (absolute, grams)', unit='g')
print()
mass_pct = percent_error_report(validation['config_id'], validation['mass_error_pct'], label='Mass (percent error)')

In [ ]:
mass_v = validation.dropna(subset=['mass_g_meas', 'mass_g_pred']).sort_values('mass_g_meas')
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))

lim = float(np.nanmax([mass_v['mass_g_meas'].max(), mass_v['mass_g_pred'].max()])) * 1.05
axes[0].scatter(mass_v['mass_g_meas'], mass_v['mass_g_pred'], s=40, color='tab:blue', edgecolor='k', linewidth=0.3)
axes[0].plot([0, lim], [0, lim], 'k--', lw=1, label='1:1')
axes[0].set_xlim(0, lim)
axes[0].set_ylim(0, lim)
axes[0].set_xlabel('measured mass [g]')
axes[0].set_ylabel('predicted mass [g]')
axes[0].set_title(f"Predicted vs measured mass\nMAE = {mass_abs['mean_abs_error']:.2f} g  |  n = {mass_abs['n']}")
axes[0].legend()
axes[0].grid(alpha=0.3)

resid = mass_v['mass_g_pred'] - mass_v['mass_g_meas']
axes[1].scatter(mass_v['mass_g_meas'], resid, s=40, color='tab:orange', edgecolor='k', linewidth=0.3)
axes[1].axhline(0, color='k', lw=0.8)
axes[1].axhline(resid.mean(), color='tab:red', ls='--', lw=1, label=f'bias = {resid.mean():+.2f} g')
axes[1].set_xlabel('measured mass [g]')
axes[1].set_ylabel('residual (pred - meas) [g]')
axes[1].set_title('Mass residuals')
axes[1].legend()
axes[1].grid(alpha=0.3)

mass_p = validation.dropna(subset=['mass_error_pct']).sort_values('mass_error_pct')
axes[2].barh(mass_p['config_id'].astype(str), mass_p['mass_error_pct'], color='tab:green', alpha=0.8)
axes[2].axvline(0, color='k', lw=0.8)
axes[2].set_xlabel('mass percent error [%]')
axes[2].set_ylabel('config_id')
axes[2].set_title(f"Per-prop mass % error\nmean |err| = {mass_pct['mean_abs_pct']:.1f}%")
axes[2].tick_params(axis='y', labelsize=6)
axes[2].grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'validation_mass.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved -> plots/nb9/validation_mass.png')

### 4.3 Stage 2 — Inertia Validation

Measured spin-axis inertia (trifilar pendulum) versus the raw STL prediction and the regression-corrected prediction (the value NB6 actually uses), with the same metrics plus the predicted-vs-measured correlation, followed by the raw-vs-corrected figures.


In [ ]:
print('=' * 64)
print('STAGE 2 - INERTIA (Izz) VALIDATION')
print('=' * 64)

print('\n--- RAW STL inertia (uncalibrated: geometry x density) ---')
izz_abs_raw = error_metrics_report(validation['config_id'], validation['izz_meas_kg_m2'], validation['izz_pred_kg_m2'], label='Inertia Izz RAW (absolute, kg*m^2)', unit='kg*m^2')
izz_pct_raw = percent_error_report(validation['config_id'], validation['izz_error_pct'], label='Inertia Izz RAW (percent error)')

print('\n--- CORRECTED inertia (linear regression a*izz_STL + b; used by NB6) ---')
izz_abs = error_metrics_report(validation['config_id'], validation['izz_meas_kg_m2'], validation['izz_pred_regressed_kg_m2'], label='Inertia Izz CORRECTED (absolute, kg*m^2)', unit='kg*m^2')
izz_pct = percent_error_report(validation['config_id'], validation['izz_error_pct_regressed'], label='Inertia Izz CORRECTED (percent error)')

keep = validation['izz_meas_kg_m2'].notna() & validation['izz_pred_kg_m2'].notna()
pear = stats.pearsonr(validation.loc[keep, 'izz_pred_kg_m2'], validation.loc[keep, 'izz_meas_kg_m2'])[0]
spear = stats.spearmanr(validation.loc[keep, 'izz_pred_kg_m2'], validation.loc[keep, 'izz_meas_kg_m2'])[0]
print(f'\nPredicted-vs-measured inertia correlation: Pearson r = {pear:.3f}, Spearman rho = {spear:.3f}')
print('(Correlation is identical for raw and corrected — the correction is a monotonic linear map.)')
print(f"\nSummary:  RAW  mean|err| = {izz_pct_raw['mean_abs_pct']:.2f}%,  bias = {izz_abs_raw['mean_error']:+.2e}")
print(f"          CORR mean|err| = {izz_pct['mean_abs_pct']:.2f}%,  bias = {izz_abs['mean_error']:+.2e}")

In [ ]:
izz_v = validation.dropna(subset=['izz_meas_kg_m2', 'izz_pred_kg_m2']).copy()
izz_v['meas_e6'] = izz_v['izz_meas_kg_m2'] * 1e6
izz_v['raw_e6'] = izz_v['izz_pred_kg_m2'] * 1e6
izz_v['regressed_e6'] = izz_v['izz_pred_regressed_kg_m2'] * 1e6
izz_v = izz_v.sort_values('meas_e6')

raw_mae = float(np.mean(np.abs(izz_v['raw_e6'] - izz_v['meas_e6'])))
reg_mae = float(np.mean(np.abs(izz_v['regressed_e6'] - izz_v['meas_e6'])))
raw_bias = float(np.mean(izz_v['raw_e6'] - izz_v['meas_e6']))
reg_bias = float(np.mean(izz_v['regressed_e6'] - izz_v['meas_e6']))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))

lim = float(np.nanmax([izz_v['meas_e6'].max(), izz_v['raw_e6'].max(), izz_v['regressed_e6'].max()])) * 1.05
axes[0].scatter(izz_v['meas_e6'], izz_v['raw_e6'], s=38, color='tab:gray', edgecolor='k', linewidth=0.3, alpha=0.7, label=f'raw STL (MAE {raw_mae:.2f})')
axes[0].scatter(izz_v['meas_e6'], izz_v['regressed_e6'], s=38, color='tab:blue', edgecolor='k', linewidth=0.3, label=f'corrected (MAE {reg_mae:.2f})')
axes[0].plot([0, lim], [0, lim], 'k--', lw=1, label='1:1')
axes[0].set_xlim(0, lim)
axes[0].set_ylim(0, lim)
axes[0].set_xlabel(r'measured $I_{zz}$ [$10^{-6}$ kg·m²]')
axes[0].set_ylabel(r'predicted $I_{zz}$ [$10^{-6}$ kg·m²]')
axes[0].set_title(f'Predicted vs measured inertia\nPearson r = {pear:.3f}  |  n = {len(izz_v)}')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].scatter(izz_v['meas_e6'], izz_v['raw_e6'] - izz_v['meas_e6'], s=36, color='tab:gray', edgecolor='k', linewidth=0.3, alpha=0.7, label=f'raw (bias {raw_bias:+.2f})')
axes[1].scatter(izz_v['meas_e6'], izz_v['regressed_e6'] - izz_v['meas_e6'], s=36, color='tab:blue', edgecolor='k', linewidth=0.3, label=f'corrected (bias {reg_bias:+.2f})')
axes[1].axhline(0, color='k', lw=0.8)
axes[1].set_xlabel(r'measured $I_{zz}$ [$10^{-6}$ kg·m²]')
axes[1].set_ylabel(r'residual [$10^{-6}$ kg·m²]')
axes[1].set_title('Inertia residuals (raw vs corrected)')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

izz_p = validation.dropna(subset=['izz_error_pct_regressed']).sort_values('izz_error_pct_regressed')
axes[2].barh(izz_p['config_id'].astype(str), izz_p['izz_error_pct_regressed'], color='tab:green', alpha=0.8)
axes[2].axvline(0, color='k', lw=0.8)
axes[2].set_xlabel('corrected inertia percent error [%]')
axes[2].set_ylabel('config_id')
axes[2].set_title(f"Per-prop inertia % error (corrected)\nmean |err| = {np.mean(np.abs(izz_p['izz_error_pct_regressed'])):.1f}%")
axes[2].tick_params(axis='y', labelsize=6)
axes[2].grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'validation_inertia.png', dpi=130, bbox_inches='tight')
plt.show()
print(f'Saved -> plots/nb9/validation_inertia.png   (raw MAE {raw_mae:.2f} -> corrected MAE {reg_mae:.2f}  [1e-6 kg m^2])')

### 4.4 Stage 3 — Load and Match Simulation vs Flight Test

Keeps the physically tested props (the NB7 representative subset plus the 10 recovered validation props), aggregates the PASS runs to one measured value per (config, RPM), and merges the aero (NB6) and release (NB6b) predictions — including their `val_` outputs, whose rows are preferred for the 10 recovered ids since they describe the true printed geometry.


In [ ]:
selected_ids = set(pd.to_numeric(pd.read_csv(SELECTED_CSV)['config_id'], errors='coerce').dropna().astype(int))

VALIDATION_PROP_IDS = set()
geometry_df = pd.read_csv(GEOMETRY_CSV)[['config_id', 'blade_count', 'radius_mm']]
if INCLUDE_VALIDATION_PROPS:
    val_geo_df = measurements.load_recovered_validation_geometry(BASE_DIR)[['config_id', 'blade_count', 'radius_mm']]
    VALIDATION_PROP_IDS = set(val_geo_df['config_id'].astype(int))
    selected_ids = selected_ids | VALIDATION_PROP_IDS
    print(f'Including {len(VALIDATION_PROP_IDS)} validation props: {sorted(VALIDATION_PROP_IDS)}')
    geometry_df = pd.concat([geometry_df[~geometry_df['config_id'].isin(val_geo_df['config_id'])], val_geo_df], ignore_index=True)

pass_runs = load_pass_runs(TEST_REPORTS)
measured_per_rpm = aggregate_pass_cells(pass_runs)

tested_ids = set(measured_per_rpm['config_id'].unique())
compared_ids = sorted(tested_ids & selected_ids)
print(f'\nRepresentative subset (NB7)  : {len(selected_ids)} props')
print(f'Tested with >=1 PASS run     : {len(tested_ids)} props')
print(f'Tested AND representative    : {len(compared_ids)} props  -> {compared_ids}')

measured_subset = measured_per_rpm[measured_per_rpm['config_id'].isin(selected_ids)].copy()

sim_aero = load_sim_long(CSV_DIR, SIM_AERO_PATTERNS, 'h_aero_m')[['config_id', 'rpm_launch', 'h_aero_m', 'can_liftoff']].rename(columns={'can_liftoff': 'aero_liftoff'})
sim_release = load_sim_long(CSV_DIR, SIM_RELEASE_PATTERNS, 'h_release_m')[['config_id', 'rpm_launch', 'h_release_m', 'h_max_aero_m', 'can_liftoff']].rename(columns={'can_liftoff': 'release_liftoff'})

matched = measured_subset.merge(sim_aero, on=['config_id', 'rpm_launch'], how='inner').merge(sim_release, on=['config_id', 'rpm_launch'], how='left').merge(geometry_df, on='config_id', how='left').sort_values(['config_id', 'rpm_launch']).reset_index(drop=True)

if VALIDATE_SIM == 'release':
    matched['h_max_m'] = matched['h_release_m']
    matched['can_liftoff'] = matched['release_liftoff']
else:
    matched['h_max_m'] = matched['h_aero_m']
    matched['can_liftoff'] = matched['aero_liftoff']
matched['sim_launch'] = matched['can_liftoff'].astype(bool)
matched['meas_launch'] = matched['meas_launch'].astype(bool)
matched['abs_err_m'] = np.where(matched['meas_censored'], np.nan, matched['h_max_m'] - matched['meas_h_max'])

print(f"\nMatched (config x launch RPM) PASS cells : {len(matched)}  across {matched['config_id'].nunique()} props")
print(f"Launch RPMs compared                     : {sorted(matched['rpm_launch'].unique())}")
print(f"Censored cells (any PASS run >= {CENSOR_AT_M} m)  : {int(matched['meas_censored'].sum())}")
matched[['config_id', 'blade_count', 'rpm_launch', 'n_pass_runs', 'h_aero_m', 'h_release_m', 'meas_h_max', 'meas_h_median']].head()

### 4.5 Stage 3b — Release Correction: Leave-One-Prop-Out Cross-Validation

Mirrors the NB6b `fit_release_correction`: OLS of the measured median height on the uncorrected aero height over the uncensored lift-off cells. Each propeller is held out in turn and predicted by a model fit on the others; with only two parameters this is not an over-fit guard but a test that the calibration generalises to an unseen propeller. Report-only — nothing downstream changes.


In [ ]:
print('=' * 72)
print('STAGE 3b - RELEASE CORRECTION: LEAVE-ONE-PROP-OUT CROSS-VALIDATION')
print('=' * 72)

calibration_cells = matched[(matched['meas_launch']) & (~matched['meas_censored'])].copy()
calibration_cells = calibration_cells.dropna(subset=['h_aero_m', 'meas_h_median'])
calibration_cells = calibration_cells[calibration_cells['h_aero_m'] > 0]

held_out_props = sorted(calibration_cells['config_id'].unique())
loo_pred = []
loo_meas = []
fold_A = []
fold_B = []
for prop_id in held_out_props:
    train = calibration_cells[calibration_cells['config_id'] != prop_id]
    test = calibration_cells[calibration_cells['config_id'] == prop_id]
    if len(train) < 2 or len(test) == 0:
        continue
    A_fold, B_fold = fit_release_AB(train)
    fold_A.append(A_fold)
    fold_B.append(B_fold)
    loo_pred.extend(A_fold + B_fold * test['h_aero_m'].to_numpy(float))
    loo_meas.extend(test['meas_h_median'].to_numpy(float))

loo_pred = np.asarray(loo_pred)
loo_meas = np.asarray(loo_meas)
loo_err = loo_pred - loo_meas

A_insample, B_insample = fit_release_AB(calibration_cells)
meas_values = calibration_cells['meas_h_median'].to_numpy(float)
insample_pred = A_insample + B_insample * calibration_cells['h_aero_m'].to_numpy(float)

try:
    production = calibration_cells.dropna(subset=['h_release_m'])
    B_prod, A_prod = np.polyfit(production['h_aero_m'], production['h_release_m'], 1)
    production_text = f'A={A_prod:.4f}  B={B_prod:.4f}  (recovered from h_release_m)'
except Exception:
    production_text = 'unavailable'

print(f"  calibration cells (uncensored, launched, aero>0): {len(calibration_cells)}  across {calibration_cells['config_id'].nunique()} props")
print(f'  folds (props held out in turn)                  : {len(fold_A)}')
print()
print(f'  Production fit (NB6b) : {production_text}')
print(f'  In-sample refit      : A={A_insample:.4f}  B={B_insample:.4f}')
print(f'     MAE = {np.abs(insample_pred - meas_values).mean():.3f} m   bias = {(insample_pred - meas_values).mean():+.3f} m')
print(f'  Leave-one-prop-out CV (out-of-sample, n={len(loo_pred)} cells):')
print(f'     MAE = {np.abs(loo_err).mean():.3f} m   bias = {loo_err.mean():+.3f} m   RMSE = {np.sqrt((loo_err ** 2).mean()):.3f} m')
print(f'     refit A across folds: {np.mean(fold_A):.4f} +/- {np.std(fold_A):.4f}')
print(f'     refit B across folds: {np.mean(fold_B):.4f} +/- {np.std(fold_B):.4f}')
print()
generalisation_gap = abs(np.abs(loo_err).mean() - np.abs(insample_pred - meas_values).mean())
if generalisation_gap < 0.05:
    print(f'  -> LOO-CV MAE within {generalisation_gap * 100:.0f} cm of in-sample: the 2-parameter correction')
    print('     generalises across props (no meaningful over-fit).')
else:
    print(f'  -> LOO-CV MAE exceeds in-sample by {generalisation_gap * 100:.0f} cm: report the absolute-accuracy')
    print('     figures as calibrated, and note this generalisation gap.')

### 4.6 Stage 3 — Global Reading

Pooled ranking (Spearman, Kendall, Pearson) and error (MAE, RMSE, bias) over all PASS, launched, uncensored cells, for both sims against both raw measured targets. Spearman/Kendall judge order (what selection needs); MAE/bias judge absolute accuracy. Aero and release share the ranking by construction; the release correction fixes the absolute error and bias.


In [ ]:
print('=' * 72)
print(f"STAGE 3 GLOBAL READING  ({matched['config_id'].nunique()} props, PASS-only)")
print('=' * 72)

rankable = matched[(matched['meas_launch']) & (~matched['meas_censored'])].copy()

global_rows = []
for target in ['meas_h_max', 'meas_h_median']:
    for sim_col, sim_name in [('h_aero_m', 'Aero (NB6)'), ('h_release_m', 'Release (NB6b)')]:
        global_rows.append(global_row(rankable, sim_col, sim_name, target))
global_reading = pd.DataFrame(global_rows)

fmt = {}
for column in ['spearman', 'kendall', 'pearson', 'bias_m']:
    fmt[column] = signed_fmt
for column in ['MAE_m', 'RMSE_m']:
    fmt[column] = plain_fmt
print(global_reading.to_string(index=False, formatters=fmt))

if USE_REGRESSED_TARGET:
    headline_target = 'meas_h_reg'
else:
    headline_target = 'meas_h_median'
headline_cells = rankable[rankable['meas_launch']].dropna(subset=['h_max_m', headline_target])
overall_rank = ranking_metrics(headline_cells['h_max_m'], headline_cells[headline_target])

print('\nGuide: Spearman/Kendall judge ORDER (selection); Pearson judges linear fit.')
print('       MAE/RMSE/bias are absolute accuracy in metres. Aero & release share ranking')
print('       (monotonic correction); release fixes the absolute error and bias.')

### 4.7 Stage 3 — Per-RPM Reading

Props are chosen at a given operating RPM, so the per-RPM ranking is the selection-relevant view. Reported for both sims against the regressed target and the median reference.


In [ ]:
for sim_col, sim_name in [('h_aero_m', 'AERO (NB6)'), ('h_release_m', 'RELEASE (NB6b)')]:
    if USE_REGRESSED_TARGET:
        target_columns = ['meas_h_reg', 'meas_h_median']
    else:
        target_columns = ['meas_h_max', 'meas_h_median']
    for target_col in target_columns:
        tab = per_rpm_table(rankable, sim_col, target_col)
        print(f"\n### {sim_name}  vs  measured {target_col.replace('meas_h_', '')} per RPM")
        print(tab.to_string(index=False, formatters=fmt))

if USE_REGRESSED_TARGET:
    per_rpm_ranking = per_rpm_table(rankable, 'h_max_m', 'meas_h_reg')
else:
    per_rpm_ranking = per_rpm_table(rankable, 'h_max_m', 'meas_h_median')

### 4.8 Stage 3 — Censored-Cell Agreement

The 2.60 m string right-censors the launcher: cells where a PASS run reached ≥ 2.40 m are excluded from all quantitative metrics (a clipped measurement has no true height). Discarding them entirely would waste real information — the rig says *this prop is a high-flyer* — so this section scores the censored cells qualitatively: does the sim also predict a ceiling-reacher?


In [ ]:
print('=' * 72)
print('STAGE 3 — CENSORED-CELL AGREEMENT (ceiling-reaching props)')
print('=' * 72)

CEILING_PREDICT_M = HEIGHT_CEILING_M

censored = matched[matched['meas_censored']].copy()
n_cens = len(censored)
print(f"\nCensored cells (a PASS run reached >= {CENSOR_AT_M} m): {n_cens}  across {censored['config_id'].nunique()} props")
print('These are EXCLUDED from the MAE / bias / ranking metrics (a clipped height has no')
print('true value). Below is the separate qualitative agreement check on them.\n')

if n_cens:
    censored['sim_predicts_ceiling'] = censored['h_aero_m'] >= CEILING_PREDICT_M
    n_agree = int(censored['sim_predicts_ceiling'].sum())
    print(f'Sim (raw aero) ALSO predicts >= {CEILING_PREDICT_M:.2f} m : {n_agree} / {n_cens}  ({100 * n_agree / n_cens:.0f}% agreement)')
    print(f"Median raw-aero height on censored cells       : {censored['h_aero_m'].median():.2f} m")
    print(f'  (a value >> {CEILING_PREDICT_M:.1f} m means the sim strongly expected a ceiling hit)\n')
    print(f"Median RELEASE-corrected height on censored cells: {censored['h_max_m'].median():.2f} m   (capped below the ceiling by the correction — expected)\n")

    summary_rows = []
    for cid, group in censored.groupby('config_id'):
        summary_rows.append({
            'config_id': int(cid),
            'n_censored_cells': len(group),
            'rpm_range': f"{int(group['rpm_launch'].min())}-{int(group['rpm_launch'].max())}",
            'median_aero_m': group['h_aero_m'].median(),
            'sim_agrees': int(group['sim_predicts_ceiling'].sum()),
        })
    summary = pd.DataFrame(summary_rows)
    summary['agree_frac'] = (summary['sim_agrees'] / summary['n_censored_cells']).round(2)
    print('Per-prop censored summary:')
    print(summary.to_string(index=False))

    launched = matched[matched['meas_launch']].copy()
    launched['meas_ceiling'] = launched['meas_censored']
    launched['sim_ceiling'] = launched['h_aero_m'] >= CEILING_PREDICT_M
    tp = int((launched['meas_ceiling'] & launched['sim_ceiling']).sum())
    fn = int((launched['meas_ceiling'] & ~launched['sim_ceiling']).sum())
    fp = int((~launched['meas_ceiling'] & launched['sim_ceiling']).sum())
    tn = int((~launched['meas_ceiling'] & ~launched['sim_ceiling']).sum())
    print('\nCeiling-reacher classification (all launched cells, aero predictor):')
    print('                     sim<ceiling   sim>=ceiling')
    print(f'  rig hit ceiling :   {fn:4d} (miss)   {tp:4d} (hit)')
    print(f'  rig below ceil. :   {tn:4d} (ok)     {fp:4d} (false alarm)')
    if tp + fn:
        recall = tp / (tp + fn)
    else:
        recall = float('nan')
    if tp + fp:
        precision = tp / (tp + fp)
    else:
        precision = float('nan')
    print(f'  recall (caught {tp}/{tp + fn} true ceiling-reachers)  = {recall:.2f}')
    print(f'  precision (of {tp + fp} flagged, {tp} truly hit cap)   = {precision:.2f}')
else:
    print('No censored cells — nothing to report.')

### 4.9 Per-Config Ranking and Trust Verdict

For each prop, how well the sim reproduces its height-vs-RPM shape (per-config Spearman), followed by a plain-language verdict on whether the pipeline can be trusted for ranking and selection.


In [ ]:
print('\n# D. PER-CONFIG: does the sim reproduce each prop\'s height-vs-RPM response?')
if USE_REGRESSED_TARGET:
    per_config_target = 'meas_h_reg'
else:
    per_config_target = 'meas_h_median'
per_config_rows = []
for cid, group in matched.groupby('config_id'):
    metrics = ranking_metrics(group['h_max_m'], group[per_config_target])
    uncensored = group[~group['meas_censored']]
    if len(uncensored):
        mae = float(np.abs(uncensored['h_max_m'] - uncensored[per_config_target]).mean())
    else:
        mae = np.nan
    if group['blade_count'].notna().any():
        blade_count = int(group['blade_count'].iloc[0])
    else:
        blade_count = -1
    per_config_rows.append({'config_id': int(cid), 'blade_count': blade_count, 'n_rpm': metrics['n'], 'spearman': metrics['spearman'], 'kendall': metrics['kendall'], 'mae_m': mae})
per_config_ranking = pd.DataFrame(per_config_rows).sort_values('spearman', ascending=False).reset_index(drop=True)
print(f'Per-config ranking computed for {len(per_config_ranking)} representative props (all of them):')
print(per_config_ranking.to_string(index=False))

median_spearman = per_config_ranking['spearman'].median()
n_strong = int((per_config_ranking['spearman'] > 0.8).sum())
n_total = int(per_config_ranking['spearman'].notna().sum())
best_row = per_config_ranking.iloc[0]
worst_row = per_config_ranking.dropna(subset=['spearman']).iloc[-1]

print('\n# E. CAN WE TRUST THE PIPELINE FOR RANKING?')
print(f'  Median per-config Spearman rho : {median_spearman:+.3f}')
print(f'  Props ranked well (rho > 0.8)  : {n_strong} / {n_total}')
print(f"  Overall pooled Spearman        : {overall_rank['spearman']:+.3f}   Kendall tau: {overall_rank['kendall']:+.3f}")
print(f"  Best-ranked prop  : config {int(best_row['config_id'])}  (rho = {best_row['spearman']:+.3f})")
print(f"  Worst-ranked prop : config {int(worst_row['config_id'])}  (rho = {worst_row['spearman']:+.3f})")
if median_spearman >= 0.7:
    verdict = 'STRONG  - the simulation reliably ranks propellers; trust it for selection.'
elif median_spearman >= 0.4:
    verdict = 'MODERATE - the simulation ranks the general order but with notable scatter; use with care.'
else:
    verdict = 'WEAK    - the simulation does not reliably rank propellers.'
print(f'  VERDICT (ranking): {verdict}')

### 4.10 Consolidated Metrics Summary

Every headline number in one place — mass, inertia, and all flight-height ranking/error metrics — written to the summary CSVs for the thesis appendix.


In [ ]:
rankable = matched[(matched['meas_launch']) & (~matched['meas_censored'])].copy()

print('#' * 78)
print('# MASTER VALIDATION SUMMARY')
print('#' * 78)

print('\n[A] SECONDARY RIGS (STL model vs bench measurement)')
secondary = pd.DataFrame([
    {'quantity': 'mass', 'n': mass_abs['n'], 'MAE': f"{mass_abs['mean_abs_error']:.3f} g",
     'mean_abs_%': f"{mass_pct['mean_abs_pct']:.2f}%", 'bias': f"{mass_abs['mean_error']:+.3f} g",
     'corr_r': '—'},
    {'quantity': 'inertia Izz', 'n': izz_abs['n'], 'MAE': f"{izz_abs['mean_abs_error']:.2e}",
     'mean_abs_%': f"{izz_pct['mean_abs_pct']:.2f}%", 'bias': f"{izz_abs['mean_error']:+.2e}",
     'corr_r': f'{pear:.3f}'},
])
print(secondary.to_string(index=False))

print('\n[B] FLIGHT HEIGHT — GLOBAL (pooled, PASS-only, uncensored)')
target_set = [('meas_h_max', 'max'), ('meas_h_median', 'median'), ('meas_h_mean', 'mean'), ('meas_h_reg', 'regressed')]
global_rows = []
for sim_col, sim_name in [('h_aero_m', 'aero'), ('h_release_m', 'release')]:
    for tcol, tname in target_set:
        if tcol not in rankable.columns:
            continue
        b = summary_block(rankable, sim_col, tcol, only_flying=(sim_col == 'h_aero_m'))
        b_row = {'sim': sim_name, 'target': tname}
        b_row.update(b)
        global_rows.append(b_row)
global_summary = pd.DataFrame(global_rows)
print(global_summary.to_string(index=False, formatters=fmt))

if USE_REGRESSED_TARGET:
    headline_target = 'meas_h_reg'
else:
    headline_target = 'meas_h_median'
print(f"\n[C] FLIGHT HEIGHT — PER-RPM (release sim vs {headline_target.replace('meas_h_', '')})")
per_rpm_rows = []
for rpm, g in rankable.groupby('rpm_launch'):
    d = g.dropna(subset=['h_release_m', headline_target])
    if d['config_id'].nunique() >= 3:
        rk = ranking_metrics(d['h_release_m'], d[headline_target])
        err = d['h_release_m'] - d[headline_target]
        per_rpm_rows.append({'rpm': int(rpm), 'n_props': rk['n'], 'spearman': rk['spearman'], 'kendall': rk['kendall'], 'MAE_m': float(np.mean(np.abs(err))), 'bias_m': float(np.mean(err))})
per_rpm_summary = pd.DataFrame(per_rpm_rows)
print(per_rpm_summary.to_string(index=False, formatters={'spearman': signed_fmt, 'kendall': signed_fmt, 'MAE_m': '{:.3f}'.format, 'bias_m': '{:+.3f}'.format}))

print('\n[D] FLIGHT HEIGHT — PER-CONFIG (release vs headline target)')
print(f"   median per-config Spearman : {per_config_ranking['spearman'].median():+.3f}")
print(f"   props well-ranked (rho>0.8): {int((per_config_ranking['spearman'] > 0.8).sum())} / {int(per_config_ranking['spearman'].notna().sum())}")
print(f"   overall pooled Spearman    : {overall_rank['spearman']:+.3f}   Kendall: {overall_rank['kendall']:+.3f}")

global_summary.to_csv(CSV_DIR / 'validation_master_summary.csv', index=False)
per_rpm_summary.to_csv(CSV_DIR / 'validation_per_rpm_summary.csv', index=False)
secondary.to_csv(CSV_DIR / 'validation_secondary_summary.csv', index=False)
print('\nSaved -> csv/validation_master_summary.csv, validation_per_rpm_summary.csv, validation_secondary_summary.csv')

### 4.11 Ranking Visualisation

The complete sim-vs-measured overview: four scatter comparisons (release vs mean/median/regressed and raw aero vs mean), rank correlation vs RPM (Spearman and Kendall), and MAE/bias vs RPM for both sims.


In [ ]:
rankable = matched[(matched['meas_launch']) & (~matched['meas_censored'])].copy()
if USE_REGRESSED_TARGET:
    target_for_plot = 'meas_h_reg'
else:
    target_for_plot = 'meas_h_median'

fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 4, hspace=0.42, wspace=0.32)

ax1 = fig.add_subplot(gs[0, 0])
scatter_panel(ax1, rankable, matched, 'h_release_m', 'meas_h_mean', 'Release-corrected vs measured MEAN')
ax2 = fig.add_subplot(gs[0, 1])
scatter_panel(ax2, rankable, matched, 'h_release_m', 'meas_h_median', 'Release-corrected vs measured MEDIAN')
ax3 = fig.add_subplot(gs[0, 2])
sc = scatter_panel(ax3, rankable, matched, 'h_release_m', 'meas_h_reg', 'Release-corrected vs monotonic FIT')
ax4 = fig.add_subplot(gs[0, 3])
scatter_panel(ax4, rankable, matched, 'h_aero_m', 'meas_h_mean', 'Raw aero vs measured MEAN', only_flying=True)
if sc is not None:
    fig.colorbar(sc, ax=[ax1, ax2, ax3, ax4], label='launch RPM', shrink=0.6, location='right', pad=0.01)

ax5 = fig.add_subplot(gs[1, :2])
for sim_col, name, style in [('h_release_m', 'release', 'o-'), ('h_aero_m', 'aero', '^--')]:
    tab = per_rpm_rank(rankable, sim_col, target_for_plot)
    if len(tab):
        ax5.plot(tab['rpm'], tab['spearman'], style, label=f'Spearman ρ ({name})')
ax5.axhline(0.7, color='grey', ls=':', lw=1, label='0.7 (good)')
ax5.axhline(0, color='k', lw=0.8)
ax5.set_ylim(-1.05, 1.05)
ax5.set_xlabel('launch RPM')
ax5.set_ylabel('rank corr across props')
if USE_REGRESSED_TARGET:
    target_name = 'monotonic fit'
else:
    target_name = 'median'
ax5.set_title(f'Ranking across props at each RPM (target: {target_name})')
ax5.legend(fontsize=8)
ax5.grid(alpha=0.3)

ax6 = fig.add_subplot(gs[1, 2:])
for sim_col, name, style in [('h_release_m', 'release', 'o-'), ('h_aero_m', 'aero', '^--')]:
    tab = per_rpm_rank(rankable, sim_col, target_for_plot)
    if len(tab):
        ax6.plot(tab['rpm'], tab['kendall'], style, label=f'Kendall τ ({name})')
ax6.axhline(0, color='k', lw=0.8)
ax6.set_ylim(-1.05, 1.05)
ax6.set_xlabel('launch RPM')
ax6.set_ylabel('Kendall τ across props')
ax6.set_title('Kendall τ across props at each RPM')
ax6.legend(fontsize=8)
ax6.grid(alpha=0.3)

ax7 = fig.add_subplot(gs[2, :2])
for sim_col, name, style in [('h_release_m', 'release', 'o-'), ('h_aero_m', 'aero', '^--')]:
    tab = per_rpm_err(rankable, sim_col, target_for_plot)
    if len(tab):
        ax7.plot(tab['rpm'], tab['mae'], style, label=f'MAE ({name})')
ax7.set_xlabel('launch RPM')
ax7.set_ylabel('MAE [m]')
ax7.set_title('Absolute height error vs RPM')
ax7.legend(fontsize=8)
ax7.grid(alpha=0.3)

ax8 = fig.add_subplot(gs[2, 2:])
for sim_col, name, style in [('h_release_m', 'release', 'o-'), ('h_aero_m', 'aero', '^--')]:
    tab = per_rpm_err(rankable, sim_col, target_for_plot)
    if len(tab):
        ax8.plot(tab['rpm'], tab['bias'], style, label=f'bias ({name})')
ax8.axhline(0, color='k', lw=0.8)
ax8.set_xlabel('launch RPM')
ax8.set_ylabel('signed bias [m]')
ax8.set_title('Height bias (sim - measured) vs RPM')
ax8.legend(fontsize=8)
ax8.grid(alpha=0.3)

fig.suptitle('Flight-height validation — ranking and error overview', fontsize=13, y=0.995)
plt.savefig(PLOTS_DIR / 'validation_ranking.png', dpi=130, bbox_inches='tight')
plt.show()

overall_frame = rankable[rankable['meas_launch']].dropna(subset=['h_max_m', target_for_plot])
overall_rank = ranking_metrics(overall_frame['h_max_m'], overall_frame[target_for_plot])
per_rpm_ranking = per_rpm_rank(rankable, 'h_max_m', target_for_plot).rename(columns={'rpm': 'rpm_launch'})
print('Saved -> plots/nb9/validation_ranking.png')

### 4.12 Per-Propeller Height vs RPM

One panel per tested propeller: the release-corrected simulated height (blue, the quantity compared to measurement), the raw aerodynamic height (green dashed, revealing the true high-RPM flyers the correction compresses), the measured median with its min-to-max spread band (orange), the smoothed monotonic-regression target (purple) and the 2.60 m string ceiling (red).


In [ ]:
all_config_ids = sorted(matched['config_id'].unique())
print(f'Plotting {len(all_config_ids)} propellers: {all_config_ids}')
has_aero = 'h_max_aero_m' in matched.columns

n_cols = 4
n_rows = int(np.ceil(len(all_config_ids) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3.2 * n_rows), squeeze=False)

for panel_index in range(n_rows * n_cols):
    ax = axes[panel_index // n_cols][panel_index % n_cols]
    if panel_index >= len(all_config_ids):
        ax.axis('off')
        continue

    config_id = all_config_ids[panel_index]
    cell = matched[matched['config_id'] == config_id].sort_values('rpm_launch')

    ax.fill_between(cell['rpm_launch'], cell['meas_h_min'], cell['meas_h_max'], color='tab:orange', alpha=0.20, label='measured run spread')
    ax.plot(cell['rpm_launch'], cell['meas_h_median'], 's-', color='tab:orange', markersize=4, label='measured median')
    if USE_REGRESSED_TARGET and 'meas_h_reg' in cell.columns:
        reg_cell = cell.dropna(subset=['meas_h_reg'])
        if len(reg_cell) >= 2:
            rpm_smooth, height_smooth = smooth_monotone_curve(reg_cell['rpm_launch'], reg_cell['meas_h_reg'])
            ax.plot(rpm_smooth, height_smooth, '-', color='tab:purple', lw=1.8, alpha=0.9, label='measured (monotonic regression)')
            ax.plot(reg_cell['rpm_launch'], reg_cell['meas_h_reg'], '.', color='tab:purple', markersize=4, alpha=0.5)
    if has_aero:
        ax.plot(cell['rpm_launch'], cell['h_max_aero_m'], '^--', color='tab:green', markersize=4, alpha=0.8, label='simulated (raw aero)')
    ax.plot(cell['rpm_launch'], cell['h_max_m'], 'o-', color='tab:blue', markersize=4, label='simulated (release-corrected)')
    ax.axhline(HEIGHT_CEILING_M, color='red', ls=':', lw=0.9)

    if cell['blade_count'].notna().any():
        blade_count = int(cell['blade_count'].iloc[0])
    else:
        blade_count = '?'
    per_prop_rho = per_config_ranking.loc[per_config_ranking['config_id'] == config_id, 'spearman']
    if len(per_prop_rho) and np.isfinite(per_prop_rho.iloc[0]):
        rho_text = f'  rho={per_prop_rho.iloc[0]:+.2f}'
    else:
        rho_text = ''
    ax.set_title(f'config {config_id} ({blade_count} blades){rho_text}', fontsize=9)
    ax.grid(alpha=0.3)
    ax.tick_params(labelsize=8)
    if panel_index == 0:
        ax.legend(fontsize=6.5, loc='upper left')
    if panel_index % n_cols == 0:
        ax.set_ylabel('peak height [m]', fontsize=8)
    if panel_index // n_cols == n_rows - 1:
        ax.set_xlabel('launch RPM', fontsize=8)

if USE_REGRESSED_TARGET:
    regression_note = 'purple = monotonic regression (ranking target); '
else:
    regression_note = ''
fig.suptitle(f'Per-propeller: simulated vs measured peak height across RPM  (all {len(all_config_ids)} representative props)\n'
             'blue = release-corrected; green dashed = raw aero; orange = measured median+spread; '
             + regression_note + 'red dotted = 2.60 m ceiling', y=1.004, fontsize=10)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'validation_per_config_height.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved plot -> plots/nb9/validation_per_config_height.png  ({len(all_config_ids)} panels, raw aero shown={has_aero})')

### 4.13 Raw Flight Trajectories

Height versus time for every (config, RPM) cell of the validated props, all repeat runs overlaid — the actual flight shape behind each aggregated point, with the simulated corrected (black) and raw-aero (green dashed) heights as reference lines.


In [ ]:
configs_traj = sorted(matched['config_id'].unique())
rpm_levels = sorted(matched['rpm_launch'].unique())
print(f'Plotting trajectories for {len(configs_traj)} configs x {len(rpm_levels)} RPM levels')

traj_idx = index_run_trajectories(TEST_CLEANED_DIRS)

n_rows = len(configs_traj)
n_cols = len(rpm_levels)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.0 * n_cols, 1.6 * n_rows), squeeze=False, sharex=False)
run_colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple']
for r, cid in enumerate(configs_traj):
    for c, rpm in enumerate(rpm_levels):
        ax = axes[r][c]
        paths = traj_idx.get((cid, int(rpm)), [])
        for k, p in enumerate(sorted(paths)):
            try:
                trace_df = pd.read_csv(p, comment='#')
            except Exception:
                continue
            if 'time_s' in trace_df.columns:
                tcol = 'time_s'
            else:
                tcol = trace_df.columns[0]
            if 'height_m' in trace_df.columns:
                hcol = 'height_m'
            else:
                hcol = trace_df.columns[-1]
            ax.plot(trace_df[tcol], trace_df[hcol], lw=0.6, color=run_colors[k % len(run_colors)], alpha=0.85)
        ax.axhline(HEIGHT_CEILING_M, color='red', ls=':', lw=0.5)
        cell = matched[(matched['config_id'] == cid) & (matched['rpm_launch'] == rpm)]
        if len(cell):
            if 'h_max_aero_m' in cell.columns and pd.notna(cell['h_max_aero_m'].iloc[0]):
                ax.axhline(cell['h_max_aero_m'].iloc[0], color='green', ls='--', lw=0.5)
            ax.axhline(cell['h_max_m'].iloc[0], color='black', lw=0.6)
        ax.tick_params(labelsize=5)
        if c == 0:
            ax.set_ylabel(f'{cid}', fontsize=6)
        if r == 0:
            ax.set_title(f'{int(rpm)}', fontsize=6)
fig.suptitle('Raw flight trajectories — height vs time (rows = config, cols = launch RPM)\n'
             'coloured = the 3 measured runs; black = sim corrected; green dashed = sim raw aero; red dotted = 2.60 m ceiling', fontsize=10, y=1.002)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'validation_trajectories.png', dpi=110, bbox_inches='tight')
plt.show()
print('Saved -> plots/nb9/validation_trajectories.png')

### 4.14 Raw RPM Traces

The companion grid: the laser-RPM channel over time for every (config, RPM) cell. This is the signal the spike filter and the spin-retention curve read from — the dashed line marks the target launch RPM, so the gap between trace and target shows the spin lost at release.


In [ ]:
n_rows = len(configs_traj)
n_cols = len(rpm_levels)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.0 * n_cols, 1.6 * n_rows), squeeze=False, sharex=False)
for r, cid in enumerate(configs_traj):
    for c, rpm in enumerate(rpm_levels):
        ax = axes[r][c]
        for k, p in enumerate(sorted(traj_idx.get((cid, int(rpm)), []))):
            try:
                trace_df = pd.read_csv(p, comment='#')
            except Exception:
                continue
            if 'rpm' not in trace_df.columns:
                continue
            if 'time_s' in trace_df.columns:
                tcol = 'time_s'
            else:
                tcol = trace_df.columns[0]
            ax.plot(trace_df[tcol], trace_df['rpm'], lw=0.6, color=run_colors[k % len(run_colors)], alpha=0.85)
        ax.axhline(rpm, color='black', ls='--', lw=0.5)
        ax.tick_params(labelsize=5)
        if c == 0:
            ax.set_ylabel(f'{cid}', fontsize=6)
        if r == 0:
            ax.set_title(f'{int(rpm)}', fontsize=6)
fig.suptitle('Raw RPM traces — rpm vs time (rows = config, cols = launch RPM)\n'
             "coloured = the 3 measured runs' laser RPM; black dashed = target launch RPM", fontsize=10, y=1.002)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'validation_rpm_traces.png', dpi=110, bbox_inches='tight')
plt.show()
print('Saved -> plots/nb9/validation_rpm_traces.png')

### 4.15 QProp Performance Surfaces

The aerodynamic engine behind the flight sim: thrust and torque operating surfaces for three representative validated propellers — a strong high-flyer, a mid prop and a weak one.


In [ ]:
SWEEP_PATH = CSV_DIR / '05_qprop_sweep.csv.gz'
SURFACE_CONFIGS = [885, 628, 507]
SURFACE_LABELS = {885: '885 (strong high-flyer)', 628: '628 (mid)', 507: '507 (weak / flat)'}

if not SWEEP_PATH.exists():
    print(f'[skip] {SWEEP_PATH.name} not found — run NB5 (QProp) first.')
else:
    sweep = pd.read_csv(SWEEP_PATH, usecols=['config_id', 'V', 'rpm', 'T', 'Q', 'qprop_ok'])
    sweep = sweep[sweep['qprop_ok']]

    fig = plt.figure(figsize=(16, 9))
    for col, cid in enumerate(SURFACE_CONFIGS):
        cfg_sweep = sweep[sweep['config_id'] == cid]
        if cfg_sweep.empty:
            continue
        for row, (value_col, zlabel, cmap) in enumerate([('T', 'thrust T [N]', 'viridis'), ('Q', 'torque Q [N·m]', 'plasma')]):
            ax = fig.add_subplot(2, 3, row * 3 + col + 1, projection='3d', computed_zorder=False)
            rpm_mesh, V_mesh, Z = make_grid(cfg_sweep, value_col)
            surf = ax.plot_surface(rpm_mesh, V_mesh, Z, cmap=cmap, edgecolor='none', alpha=0.95)
            ax.set_xlabel('RPM', fontsize=8, labelpad=2)
            ax.set_ylabel('V [m/s]', fontsize=8, labelpad=2)
            ax.set_zlabel(zlabel, fontsize=8, labelpad=2)
            ax.tick_params(labelsize=7)
            ax.view_init(elev=24, azim=-128)
            z_min = float(np.nanmin(Z))
            z_max = float(np.nanmax(Z))
            if z_max > z_min:
                z_span = z_max - z_min
            else:
                z_span = 1.0
            ax.set_zlim(z_min - 0.30 * z_span, z_max + 0.05 * z_span)
            for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
                axis.pane.set_visible(False)
            ax.grid(False)
            if row == 0:
                ax.set_title(f'config {SURFACE_LABELS.get(cid, cid)}', fontsize=10, pad=2)
            fig.colorbar(surf, ax=ax, shrink=0.5, aspect=12, pad=0.08)
    fig.suptitle('QProp operating surfaces — thrust (top) and torque (bottom) vs (RPM, forward speed)', fontsize=13, y=0.99)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(PLOTS_DIR / 'validation_qprop_surfaces.png', dpi=130, bbox_inches='tight')
    plt.show()
    print('Saved -> plots/nb9/validation_qprop_surfaces.png')

### 4.16 Validation on Void-Free Cells Only

QProp fails to converge at high advance ratio, leaving voids in each prop's T(V, ω) surface that NB6 fills by nearest-neighbour. This recomputes the metrics on only the cells whose flight stays inside the void-free region (V ≤ 5 m/s, ω ≤ launch RPM); if they match the all-cells result, the void fill is not biasing the validation.


In [ ]:
SWEEP_PATH = CSV_DIR / '05_qprop_sweep.csv.gz'
if not SWEEP_PATH.exists():
    print(f'[skip] {SWEEP_PATH.name} not found.')
else:
    sweep_all = pd.read_csv(SWEEP_PATH, usecols=['config_id', 'V', 'rpm', 'T', 'Q', 'qprop_ok'])
    sweep_all = sweep_all[(sweep_all['qprop_ok']) & (sweep_all['config_id'].isin(matched['config_id'].unique()))]

    matched_clean = matched.copy()
    clean_flags = []
    for c, rp in zip(matched_clean['config_id'], matched_clean['rpm_launch']):
        clean_flags.append(cell_is_clean(sweep_all, int(c), int(rp)))
    matched_clean['surface_clean'] = clean_flags
    rankable_all = matched_clean[(matched_clean['meas_launch']) & (~matched_clean['meas_censored'])]
    rankable_clean = rankable_all[rankable_all['surface_clean']]
    if USE_REGRESSED_TARGET:
        headline_target = 'meas_h_reg'
    else:
        headline_target = 'meas_h_median'

    print('=' * 64)
    print('VALIDATION ON VOID-FREE CELLS ONLY')
    print('=' * 64)
    print(f'Rankable cells (all)       : {len(rankable_all)}')
    print(f'Rankable cells (void-free) : {len(rankable_clean)} ({100 * len(rankable_clean) / max(len(rankable_all), 1):.0f}%)')

    rows = []
    for label, frame in [('all', rankable_all), ('void-free', rankable_clean)]:
        for sim_col, nm in [('h_aero_m', 'aero'), ('h_release_m', 'release')]:
            b = void_block(frame, sim_col, headline_target)
            if b:
                b_row = {'subset': label, 'sim': nm}
                b_row.update(b)
                rows.append(b_row)
    print()
    print(pd.DataFrame(rows).to_string(index=False))
    print('\nIf void-free matches all-cells, the nearest-neighbour void fill is not distorting the result.')

### 4.17 Liftoff-Classification Accuracy vs RPM

Beyond how high, did it fly at all? The aero model predicts lift-off only when the prop can aerodynamically hover; the release model includes the ballistic kick of the screw release, which lifts props that cannot hover.


In [ ]:
LIFTOFF_THRESHOLD_M = LIFTOFF_HEIGHT_M

clf = matched.dropna(subset=['h_aero_m', 'h_release_m']).copy()
clf['meas_fly'] = clf['meas_launch'].astype(bool)
clf['aero_fly'] = clf['h_aero_m'] > LIFTOFF_THRESHOLD_M
clf['release_fly'] = clf['h_release_m'] >= LIFTOFF_THRESHOLD_M

rows = []
for rpm, g in clf.groupby('rpm_launch'):
    rows.append({
        'rpm': int(rpm), 'n': len(g),
        'rig_flew_frac': g['meas_fly'].mean(),
        'aero_accuracy': (g['aero_fly'] == g['meas_fly']).mean(),
        'release_accuracy': (g['release_fly'] == g['meas_fly']).mean(),
    })
liftoff_acc = pd.DataFrame(rows).sort_values('rpm')

print('=' * 64)
print('LIFTOFF CLASSIFICATION ACCURACY vs RPM')
print('=' * 64)
print(liftoff_acc.to_string(index=False, formatters={'rig_flew_frac': '{:.2f}'.format, 'aero_accuracy': '{:.2f}'.format, 'release_accuracy': '{:.2f}'.format}))
print(f"\nOverall: aero = {(clf['aero_fly'] == clf['meas_fly']).mean():.2f}, release = {(clf['release_fly'] == clf['meas_fly']).mean():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
axes[0].plot(liftoff_acc['rpm'], liftoff_acc['aero_accuracy'] * 100, '^--', color='tab:green', lw=2, label='aero (NB6)')
axes[0].plot(liftoff_acc['rpm'], liftoff_acc['release_accuracy'] * 100, 'o-', color='tab:blue', lw=2, label='release (NB6b)')
axes[0].set_ylim(0, 105)
axes[0].set_xlabel('launch RPM')
axes[0].set_ylabel('liftoff classification accuracy [%]')
axes[0].set_title('Liftoff accuracy vs RPM\n(release kick fixes the low-RPM no-fly errors)')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

axes[1].plot(liftoff_acc['rpm'], liftoff_acc['rig_flew_frac'] * 100, 's-', color='black', lw=2, label='rig actually flew')
agg_pred = clf.groupby('rpm_launch').agg(aero=('aero_fly', 'mean'), rel=('release_fly', 'mean')).reset_index()
axes[1].plot(agg_pred['rpm_launch'], agg_pred['aero'] * 100, '^--', color='tab:green', lw=2, label='aero predicts fly')
axes[1].plot(agg_pred['rpm_launch'], agg_pred['rel'] * 100, 'o-', color='tab:blue', lw=2, label='release predicts fly')
axes[1].set_ylim(0, 105)
axes[1].set_xlabel('launch RPM')
axes[1].set_ylabel('fraction predicted / measured to fly [%]')
axes[1].set_title('Fraction flying vs RPM\n(aero under-predicts liftoff at low RPM)')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'validation_liftoff_accuracy.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved -> plots/nb9/validation_liftoff_accuracy.png')

### 4.18 Evidence for the Screw-Release Mechanism

Two independent measurements support the release model: the spin-retention curve (the unscrew strips most of the spin, increasingly so at higher RPM) and the cells where the aero sim says the prop cannot hover (T/W < 1) yet the rig still flew — their measured heights cluster near the ballistic floor of a ~1.9 m/s kick (v₀²/2g ≈ 0.18 m).


In [ ]:
RETENTION_CSV = CSV_DIR / '06b_release_retention_curve.csv'

nonhover = matched[(matched['aero_liftoff'] == False) & (matched['meas_launch'])].copy()
nonhover = nonhover.dropna(subset=['meas_h_median'])
band = nonhover[(nonhover['meas_h_median'] >= LIFTOFF_HEIGHT_M) & (nonhover['meas_h_median'] <= 0.8)]

print(f'Non-hover (T/W<1) cells that still flew : {len(band)}')
print(f"  median measured height : {band['meas_h_median'].median() * 100:.0f} cm")
print(f'  ballistic kick floor   : v0={KICK_VELOCITY_MS} m/s -> {H_KICK_FLOOR * 100:.0f} cm')

fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 5))

if RETENTION_CSV.exists():
    ret = pd.read_csv(RETENTION_CSV)
    axL.plot(ret['target_rpm'], ret['retention'] * 100, 'o-', color='tab:blue', lw=2.2, markersize=7)
    axL.fill_between(ret['target_rpm'], 0, ret['retention'] * 100, color='tab:blue', alpha=0.08)
    axL.annotate(f"{ret['retention'].iloc[0] * 100:.0f}%", (ret['target_rpm'].iloc[0], ret['retention'].iloc[0] * 100), textcoords='offset points', xytext=(8, 8), fontsize=10, color='tab:blue', fontweight='bold')
    axL.annotate(f"{ret['retention'].iloc[-1] * 100:.0f}%", (ret['target_rpm'].iloc[-1], ret['retention'].iloc[-1] * 100), textcoords='offset points', xytext=(-5, 12), fontsize=10, color='tab:blue', fontweight='bold')
    axL.set_ylim(0, 100)
else:
    axL.text(0.5, 0.5, 'retention curve CSV not found', ha='center', transform=axL.transAxes)
axL.set_xlabel('target / launch RPM')
axL.set_ylabel('RPM retained in flight [%]')
axL.set_title('(a)  Spin retention vs launch RPM')
axL.grid(alpha=0.3)

axR.hist(band['meas_h_median'], bins=24, color='tab:green', alpha=0.8, edgecolor='white', linewidth=0.4)
axR.axvline(H_KICK_FLOOR, color='tab:red', lw=2, ls='--', label=f'ballistic kick v₀={KICK_VELOCITY_MS} m/s → {H_KICK_FLOOR * 100:.0f} cm')
axR.axvline(band['meas_h_median'].median(), color='black', lw=1.4, ls=':', label=f"median = {band['meas_h_median'].median() * 100:.0f} cm")
axR.set_xlabel('measured peak height [m]')
axR.set_ylabel('number of (config, RPM) cells')
axR.set_title(f'(b)  Designs that cannot hover (T/W<1) still fly\n{len(band)} cells, sim predicts no lift-off')
axR.legend(fontsize=9)
axR.grid(alpha=0.3)

fig.suptitle('Evidence for the screw-release mechanism', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'validation_release_evidence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> plots/nb9/validation_release_evidence.png')

### 4.19 Save Validation Tables

In [ ]:
CSV_DIR.mkdir(parents=True, exist_ok=True)
saved = []
try:
    validation[['config_id', 'mass_g_meas', 'mass_g_pred', 'izz_meas_kg_m2', 'izz_pred_kg_m2', 'izz_pred_regressed_kg_m2']].to_csv(CSV_DIR / 'validation_mass_inertia.csv', index=False)
    saved.append('validation_mass_inertia.csv')
except Exception as e:
    print('  [warn] mass/inertia save:', e)
try:
    per_config_ranking.to_csv(CSV_DIR / 'validation_sim_ranking_per_config.csv', index=False)
    saved.append('validation_sim_ranking_per_config.csv')
except Exception as e:
    print('  [warn] per-config save:', e)
try:
    matched.to_csv(CSV_DIR / 'validation_sim_matched.csv', index=False)
    saved.append('validation_sim_matched.csv')
except Exception as e:
    print('  [warn] matched save:', e)
print('Saved:', ', '.join(saved))

### 4.20 Validated Dataset

Assembles `csv/dataset_validated.csv` — the measured counterpart of the full simulation dataset (`csv/dataset_full_simulation.csv`, built by NB6b). Same column set, one row per tested (config, RPM) cell, with **measured** values wherever a measurement exists: the true printed geometry (from `csv/00_validation_geometry.csv`), the bench-measured mass and trifilar inertia, and the launcher-measured peak height (the isotonic-regressed PASS-mean as `h_max_m`, with median/mean/min/max, run counts and censoring flags alongside). Fields that only exist in simulation (static thrust, flight time, speeds) are left empty rather than filled with simulated numbers; the simulated release-corrected and raw-aero heights are appended as clearly named reference columns. The table regenerates automatically as more propellers are tested — new PASS runs and new bench measurements simply add rows.


In [ ]:
geometry_full = measurements.load_validation_geometry(BASE_DIR)
geo_columns = ['config_id', 'radius_mm', 'ring_height_mm', 'ring_thickness_mm', 'blade_count',
               'inner_thickness_pct', 'inner_max_pos', 'inner_camber', 'inner_chord_mm', 'inner_angle_deg',
               'mid_radial_pos', 'mid_chord_mm', 'mid_angle_deg',
               'outer_thickness_pct', 'outer_max_pos', 'outer_camber', 'outer_chord_mm', 'outer_angle_deg']

confidence_frames = []
for name in ['04_xfoil_polars.csv', 'val_04_xfoil_polars.csv']:
    path = CSV_DIR / name
    if path.exists():
        confidence_frames.append(pd.read_csv(path, usecols=['config_id', 'confidence_score']))
confidence_df = pd.concat(confidence_frames, ignore_index=True).drop_duplicates('config_id', keep='last')

dataset = matched.merge(geometry_full[geo_columns + ['geometry_source']], on='config_id', how='left', suffixes=('_matched', ''))
dataset = dataset.merge(validation[['config_id', 'mass_g_meas', 'izz_meas_kg_m2']], on='config_id', how='left')
dataset = dataset.merge(confidence_df, on='config_id', how='left')

validated = pd.DataFrame()
validated['config_id'] = dataset['config_id'].astype(int)
validated['rpm_launch'] = dataset['rpm_launch'].astype(int)
for column in geo_columns:
    if column != 'config_id':
        validated[column] = dataset[column]
validated['mass_g'] = dataset['mass_g_meas']
validated['izz_kg_m2'] = dataset['izz_meas_kg_m2']
validated['frontal_area_drag_m2'] = np.nan
validated['blade_planform_m2'] = np.nan
validated['confidence_score'] = dataset['confidence_score']
validated['stl_ok'] = True
validated['qprop_ok'] = np.nan
validated['flight_ok'] = np.nan
validated['can_liftoff'] = dataset['meas_launch'].astype(bool)
validated['T_static_N'] = np.nan
validated['Q_static_Nm'] = np.nan
validated['Pshaft_static_W'] = np.nan
validated['T_over_W'] = np.nan
validated['h_max_aero_m'] = np.nan
validated['h_max_m'] = dataset['meas_h_reg']
validated['flight_time_s'] = np.nan
validated['hover_time_s'] = np.nan
validated['v_max_m_s'] = np.nan
validated['v_impact_m_s'] = np.nan
validated['rpm_at_impact'] = np.nan
validated['data_source'] = 'measured'
validated['geometry_source'] = dataset['geometry_source']
validated['meas_h_median'] = dataset['meas_h_median']
validated['meas_h_mean'] = dataset['meas_h_mean']
validated['meas_h_min'] = dataset['meas_h_min']
validated['meas_h_max'] = dataset['meas_h_max']
validated['n_pass_runs'] = dataset['n_pass_runs']
validated['meas_censored'] = dataset['meas_censored']
validated['h_sim_release_m'] = dataset['h_release_m']
validated['h_sim_aero_m'] = dataset['h_aero_m']

validated = validated.sort_values(['config_id', 'rpm_launch']).reset_index(drop=True)
validated.to_csv(CSV_DIR / cfg.CSV_NAMES['dataset_validated'], index=False)
print(f"Saved -> {cfg.CSV_NAMES['dataset_validated']}  ({len(validated)} rows, {validated['config_id'].nunique()} props, {validated.shape[1]} columns)")
validated.head()

### 4.21 How to Read These Results

- **Stage 1 / 2 (mass, inertia)** validate the secondary rigs: STL-derived mass and inertia vs the scale and trifilar pendulum. Both raw and regression-corrected inertia are reported.
- **Stage 3 (flight height)** is the core. It compares the **aero** (NB6) and **release-corrected** (NB6b) simulations against the launcher, using **PASS-only, spike-filtered** measurements over all 30 tested props.
- **Three ranking views**: *global* (pooled, weakest), *per-RPM* (selection-relevant — ranks props at a fixed operating RPM), and *per-config* (does the sim reproduce one prop's height-vs-RPM shape). The per-config view is the strongest; the release correction fixes the absolute magnitude without touching any ranking.
- **Censored cells** (string-ceiling hits) are excluded from every quantitative metric and scored separately for qualitative agreement.
- The validated dataset `csv/dataset_validated.csv` carries the measured values in the same schema as the full simulation dataset, ready for data-driven modelling and for future test batches.
